# Demo Self-Attention và Causal Mask bằng PyTorch

Bài tập thực hành môn Deep Learning: Hiện thực hóa cơ chế Self-Attention từ đầu bằng PyTorch, trực quan hóa attention heatmap trước và sau Causal Mask, và đo lường chi phí tính toán thực tế trên GPU và CPU.

Dùng **trọng số đã huấn luyện của GPT-2** (bản `gpt2`, 124M tham số), không phải trọng số ngẫu nhiên.

In [1]:
import platform
import torch
from transformers import GPT2Model, GPT2TokenizerFast

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: Không có (chạy trên CPU)")
print(f"CPU: {platform.processor()}")

C:\Users\Legion\Desktop\TÀI LIỆU HỌC\Presenttation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1113 13:31:45.094000 32212 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch version: 2.9.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CPU: AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD


## Tải mô hình GPT-2 tiền huấn luyện

Tải mô hình `GPT2Model` và `GPT2TokenizerFast` từ Hugging Face. Mô hình được đặt ở chế độ đánh giá (`eval()`) với kiểu dữ liệu `float32` và `attn_implementation="eager"` để trích xuất ma trận attention gốc làm đối chứng.

In [2]:
print("Đang tải mô hình GPT-2...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2", attn_implementation="eager")
model.eval()
model.to(DEVICE)
print("Đã tải xong mô hình GPT-2 thành công.")

Đang tải mô hình GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 36998.27it/s]

Đã tải xong mô hình GPT-2 thành công.


## Hàm Self-Attention tự viết (From Scratch)

Công thức tính toán cơ chế Scaled Dot-Product Attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V$$

Trong đó:
- $Q, K, V$ được chiếu từ hidden state $X$ thông qua ma trận trọng số $W_{qkv}$ và bias $b_{qkv}$ của lớp `c_attn`.
- $d_k = 64$ là chiều của mỗi đầu attention ($768 / 12$).
- $M$ là ma trận Causal Mask: $M_{i,j} = 0$ khi $j \le i$ (quá khứ và hiện tại), và $M_{i,j} = -\infty$ khi $j > i$ (tương lai).

> **Quy tắc bắt buộc**: Hàm `self_attention` dưới đây được tự viết từng bước phép tính ma trận bằng PyTorch, không gọi `scaled_dot_product_attention`, `nn.MultiheadAttention` hay module attention có sẵn của GPT-2.

In [3]:
# Câu ví dụ chuẩn theo thiết kế
SENTENCE = "The animal didn't cross the street because it was too tired"
input_ids = tokenizer.encode(SENTENCE, return_tensors="pt").to(DEVICE)
raw_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
tokens = [t.replace('Ġ', '') for t in raw_tokens]
print("Danh sách tokens:", tokens)

def get_layer_input(model, input_ids, layer: int):
    """Lấy hidden state của lớp đã chọn sau LayerNorm 1 (ln_1)"""
    hidden_states = model(input_ids, output_hidden_states=True).hidden_states[layer][0]
    return model.h[layer].ln_1(hidden_states)

def get_qkv_weights(model, layer: int):
    """Lấy trọng số chiếu QKV từ checkpoint GPT-2 đã huấn luyện"""
    return model.h[layer].attn.c_attn.weight, model.h[layer].attn.c_attn.bias

def self_attention(x, w_qkv, b_qkv, n_head: int = 12, causal: bool = True):
    """
    Hàm tính self-attention tự viết từng bước bằng PyTorch.
    Trả về: (đầu ra out đã ghép các head trước c_proj, ma trận attention shape (n_head, T, T))
    """
    T, C = x.shape
    d = C // n_head
    
    # Bước 1: Chiếu tuyến tính ra QKV bằng phép nhân ma trận x @ w_qkv + b_qkv, sau đó tách Q, K, V
    qkv = x @ w_qkv + b_qkv
    q, k, v = qkv.chunk(3, dim=-1)
    
    # Bước 2: Chia thành n_head đầu chú ý, mỗi đầu có kích thước d = 64
    q = q.view(T, n_head, d).transpose(0, 1)  # (n_head, T, d)
    k = k.view(T, n_head, d).transpose(0, 1)  # (n_head, T, d)
    v = v.view(T, n_head, d).transpose(0, 1)  # (n_head, T, d)
    
    # Bước 3: Tính điểm tương đồng bằng tích vô hướng Q @ K^T chia căn bậc hai của d
    scores = torch.matmul(q, k.transpose(-1, -2)) / (d ** 0.5)  # (n_head, T, T)
    
    # Bước 4: Nếu causal=True, gán -inf vào các vị trí tương lai (nửa trên đường chéo)
    if causal:
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
    
    # Bước 5: Áp dụng Softmax để ra trọng số chú ý (tổng mỗi hàng bằng 1) và nhân với V
    attn = torch.softmax(scores, dim=-1)  # (n_head, T, T)
    out = torch.matmul(attn, v)  # (n_head, T, d)
    out = out.transpose(0, 1).contiguous().view(T, C)  # (T, C)
    
    return out, attn


Danh sách tokens: ['The', 'animal', 'didn', "'t", 'cross', 'the', 'street', 'because', 'it', 'was', 'too', 'tired']


## Kiểm tra khớp với GPT-2 gốc (Verification & Assertions)

So sánh ma trận attention do hàm tự viết sinh ra với ma trận attention chính thức do `GPT2Model` của Hugging Face trả về trên toàn bộ 12 layer. Sai số lớn nhất phải nhỏ hơn $10^{-4}$.

In [4]:
with torch.no_grad():
    ref = model(input_ids, output_attentions=True).attentions  # tuple 12 lớp, mỗi lớp (1, 12, T, T)
    max_diff = 0.0
    for layer in range(12):
        x = get_layer_input(model, input_ids, layer)
        _, attn = self_attention(x, *get_qkv_weights(model, layer), causal=True)
        diff = (attn - ref[layer][0]).abs().max().item()
        assert diff < 1e-4, f"Lớp {layer}: lệch {diff:.2e}"
        max_diff = max(max_diff, diff)
    
    # Kiểm tra tính chất toán học của phân phối xác suất softmax
    assert torch.allclose(attn.sum(-1), torch.ones_like(attn.sum(-1)), atol=1e-5), "Tổng hàng softmax phải bằng 1"
    # Kiểm tra tính chất triệt tiêu tương lai của causal mask
    assert attn.triu(1).abs().max() == 0, "Nửa trên đường chéo sau mask phải hoàn toàn bằng 0"

print(f"✓ Hàm tự viết khớp GPT-2 gốc ở cả 12 lớp!")
print(f"✓ Sai số lớn nhất: {max_diff:.2e} (ngưỡng cho phép: < 1.00e-04)")
print(f"✓ Tổng hàng ma trận chú ý = 1.0; Tam giác trên sau causal mask = 0.0")


✓ Hàm tự viết khớp GPT-2 gốc ở cả 12 lớp!
✓ Sai số lớn nhất: 1.76e-06 (ngưỡng cho phép: < 1.00e-04)
✓ Tổng hàng ma trận chú ý = 1.0; Tam giác trên sau causal mask = 0.0
